## Neural network models by hand

Here we build and solve a simple neural network by hand: this means setting up all matrices, then doing forward propagation to get model weights and make predictions.

For this, we won't need many libraries, `numpy` (matrix algebra) will be enough.

In [ ]:
## import libraries
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_diabetes


In [ ]:
np.random.seed(43)

### Data

We use the built-in `diabetes` dataset from `scikit-learn`.
We take a subset of examples and features:

In [ ]:
X, y = load_diabetes(return_X_y=True)

rng = np.random.default_rng(seed=42)

n = 40
idx = rng.choice(len(X), size=n, replace=False)

X = X[idx]
y = y[idx]

## we take four variables: age, sex, BMI, serum cholesterol
X = X[:,[0,1,2,4]]

In [ ]:
print("size of the feature data matrix:", X.shape)
print("size of the target vector:", y.shape)

In [ ]:
print(X[0:10,:])

- <font color='yellow'>**age: age in years**</font>
- <font color='yellow'>**sex**</font>
- <font color='yellow'>**bmi: body mass index**</font>
- bp: average blood pressure
- <font color='yellow'>**s1: tc, total serum cholesterol**</font>
- s2: ldl, low-density lipoproteins
- s3: hdl, high-density lipoproteins
- s4: tch, total cholesterol / HDL
- s5: ltg, possibly log of serum triglycerides level
- s6: glu, blood sugar level

In [ ]:
print(y)

## Neural Network model

In the previous exercise we used logistic regression implemented as a simple neural network model with just the output layer: this output layer had only one node (binary classification) which performed both the regression (linear combination of input variables + bias) and sigmoid activation steps.

We now move on to a wee neural network with two layers and three nodes overall:

- hidden layer: two units
- output layer: one unit

We start with the first half of the first epoch: the first **forward propagation pass**

### Forward propagation

#### Layer 1 (hidden layer)

<img src="https://drive.google.com/uc?export=view&id=1HDVb_J_dqauxyhMtnCty6crrqqpKtdRz">

These are the matrices that we need in the first (hidden) layer (!! remember: neural networks are a series of **matrix algebra operations** - a lot of them !!):

- **X** (n,m): the input data
- **W1** (m,u): the weights (model parameters)
- **b1** (1,u): the bias terms (more model parameters)


In [ ]:
n = len(X)
m = X.shape[1]
u = 2

In [ ]:
print("N. of samples:", n)
print("N. of features:", m)
print("N. of units:", u)

We use He/Kaiming initialization (https://www.geeksforgeeks.org/deep-learning/kaiming-initialization-in-deep-learning/: this should reduce problems with zero activations / dead neurons (which can happen, especially with small problems, e.g. few features, few unints):

In [ ]:
## FUNCTIONS
def he_initialization(n_in, n_out):
    """
    Initializes weights using He (Kaiming) normal distribution
    and sets biases to a small positive value (0.01).
    n_in: input features
    n_out: output units
    """
    # Standard deviation is sqrt(2 / n_in)
    stddev = np.sqrt(2.0 / n_in)

    # Sample weights from a standard normal distribution and scale
    weights = np.random.randn(n_in, n_out) * stddev

    # Initialize biases to 0.01 instead of 0 to avoid dead ReLUs
    biases = np.ones((1, n_out)) * 0.01

    return weights, biases

In [ ]:
W1, b1 = he_initialization(m,u)

In [ ]:
print(W1)

In [ ]:
print(b1)

We can now calculate the linear combinations of the input features in layer 1:

In [ ]:
Z1 = np.matmul(X,W1) + b1

In [ ]:
Z1.shape

In [ ]:
print(Z1)

We activate the linear combination of input features in the hidden layer by using the ReLU activation function:

In [ ]:
A = np.maximum(Z1, 0)

In [ ]:
A

#### Layer 2 (output layer)

<img src="https://drive.google.com/uc?export=view&id=1O3dlPAkPYGE-KBDuA0VyvZfUTIKw7f6J">

- **A** (n,u)
- **W2** (u,1)
- **b2** (1,1)


In [ ]:
W2, b2 = he_initialization(u,1)

In [ ]:
W2

In [ ]:
b2

In [ ]:
Z2 = np.matmul(A,W2) + b2

In [ ]:
Z2

#### Predictions and loss

In [ ]:
y_hat =  np.copy(Z2)

In [ ]:
def loss(y, t):
    """MSE loss function"""
    return np.mean((t - y)**2)

In [ ]:
loss_value = loss(y, y_hat)

In [ ]:
print('Loss after first (half) epoch:',round(loss_value, 2))

### Backward propagation

#### Only one unit

The details of a full backprogation pass would be too complex and cumbersome.
Therefore, we simplify this brutally and only use one unit (the output layer) and one feature for illustration:

- single unit of the output layer
- take one of the two activation values from the hidden layer (new "features" for the output layer)

To illustrate backpropagation, we need to encapsulate all code into functions (remember that we will need to take partial derivatives of functions: this is the essence of backpropagation):


In [ ]:
## function to calculate the output of the simplified network
def nn(x, w):
    """Output function y = x * w"""
    return x * w

We now take the first activation feature and the corresponding weight:

In [ ]:
feat_ind = 0

x = A[:,feat_ind] ## one feature
w = W2[feat_ind] ## corresponding weight

nn(x,w)

In gradient descent, we update parameters in the following way:

$$
w_{t+1} = w_t - \alpha \cdot \Delta(w)
$$

where $\alpha$ is the learning rate and $\Delta(w)$ is the **gradient**:

$$
\Delta(w) = \frac{\partial L}{\partial w}
$$

i.e. the partial derivative of the loss function with respect to the parameter $w$.

By the **chain rule**:

$$
\frac{\partial L}{\partial w} = \frac{\partial L}{\partial \hat{y}} \cdot \frac{\partial \hat{y}}{\partial w}
$$

We are using the squared loss:

$$
\frac{\partial L}{\partial \hat{y}} = \frac{\partial (y - \hat{y})^2}{\partial \hat{y}} = -2(y - \hat{y}) = 2(\hat{y} - y)
$$

Since $\hat{y} = w \cdot x$:

$$
\frac{\partial \hat{y}}{\partial w} = \frac{\partial (x \cdot w)}{\partial w} = x
$$

Hence:

$$
\Delta(w) = \frac{\partial L}{\partial w} = 2x \cdot (\hat{y} - y)
$$

In [ ]:
def gradient(w, x, y):
    """Gradient function. (Remember that y_hat = nn(x, w) = x * w)"""
    return 2 * x * (nn(x, w) - y)


In [ ]:
def delta_w(w_t, x, y, learning_rate):
    """Update function delta w"""
    return learning_rate * np.mean(gradient(w_t, x, y))

In [ ]:
learning_rate = 0.95
n_iterations = 1  # number of gradient descent updates

In [ ]:
# Keep track of weight and loss values
w_loss = [(w, loss(nn(x, w), y))]
w_loss

In [ ]:
for i in range(n_iterations):
    dw = delta_w(w, x, y, learning_rate)  # Delta w update
    w = w - dw  # Update the current weight parameter
    w_loss.append((w, loss(nn(x, w), y)))  # Save weight and loss

In [ ]:
# Print the final w, and loss
for i in range(0, len(w_loss)):
    print(f'w({i}): {w_loss[i][0].item():.4f} \t loss: {w_loss[i][1]:.4f}')

Plot of gradient descent

In [ ]:
upper_bound = 20
ws = np.linspace(0, upper_bound, num=100)  # weight values
loss_ws = np.vectorize(lambda w: loss(nn(x, w) , y))(ws)

plt.figure(figsize=(6, 4))
plt.plot(ws, loss_ws, 'r--', label='loss')  # Loss curve
# Plot the updates
for i in range(0, len(w_loss)-1):
    w1, c1 = w_loss[i]
    w2, c2 = w_loss[i+1]
    plt.plot(w1, c1, 'bo')
    plt.plot([w1, w2],[c1, c2], 'b-')
    plt.text(w1, c1+0.05, f'${i}$')
plt.plot(w2, c2, 'bo', label='$w(k)$')
plt.text(w2, c2+0.05, f'${i+1}$')
# Show figure
plt.xlabel('$w$', fontsize=12)
plt.ylabel('$\\xi$', fontsize=12)
plt.title('Gradient descent updates plotted on loss function')
plt.xlim(0, upper_bound)
plt.legend(loc=1)
plt.show()

In [ ]:
W2

In [ ]:
W2_2 = np.copy(W2)
W2_2[0] = w_loss[1][0]

In [ ]:
W2_2

In [ ]:
Z2_2 = np.matmul(A,W2_2) + b2

In [ ]:
y_hat_2 =  np.copy(Z2_2)

In [ ]:
loss_value_2 = loss(y, y_hat_2)

In [ ]:
print(round(loss_value, 2))
print(round(loss_value_2, 2))

### Backpropagation

https://peterroelants.github.io/posts/neural-network-implementation-part01/

https://peterroelants.github.io/posts/neural-network-implementation-part03/



<font color='yellow'>**Question for you: what is the size of the input matrix $\mathbf{X}$, the matrix of coefficients $\mathbf{W}$ and the vector of bias terms $\mathbf{b}$ in the hidden layer (layer n. 1)?**</font>

We chose <a href='https://towardsdatascience.com/understanding-binary-cross-entropy-log-loss-a-visual-explanation-a3ac6025181a'>binary cross-entropy</a> for binary classification problems: this is the loss function we have been discussing in the slides.
As for the optimizer, we use `rmsprop`: more details on the available otimizers in Keras can be found <a href='https://keras.io/api/optimizers/'>here</a>

<font color='yellow'>**Question for you: what does the `input_shape` indicate?**</font>

#### Training and test sets

We now prepare the training and test set for correct evaulation of the neural network model performance: we make one split (one training set and one test set), and assign 80% of the data to the training set and 20% of the data to the test set

#### Building the neural network model

We now build the (shallow) **neural network model** using the `Keras` framework: we choose a `sequential` architecture and add `dense` (fully connected) layers (**1 hidden layer**, **1 output layer**).

We chose `ReLU` activation functions for the units in the hidden layer, and `sigmoid` activation function for the output layer.



- going deeper
- dl for logistic regression is a bit of an overkill (Ng's slide below)

## A deeper neural networks model



As a matter of fact, a reliable way to get better predictive performance is often to either **train a bigger network** or feed **more data** to it.
Eventually you'll hit the limit: i) run out of training examples; or ii) network so big that it is too slow to train